# 🧪 Lab 2: The Internal Engine Diagnostics (The Reference Tracking Boundaries)

Welcome to the internal autopsy bay. In this lab, we isolate and expose the exact structural boundaries of Kryo's internal reference tracker.

**Mission Objective:** We test two distinct object architectures to map where Kryo can deduplicate data and where it fails. 
1. **Workload A (Intra-Record Deduplication):** A single heavy 5KB metadata object pointer is duplicated 10 times *within the exact same row graph*. Reference tracking should compress this instantly.
2. **Workload B (Inter-Record / The No-Deduplication Illusion):** A single heavy 5KB metadata object pointer is shared *across 5,000 independent rows*. This will prove that Kryo resets its memory between rows during a shuffle, offering zero cross-record deduplication even with tracking active.


### Step 1: Define the Low-Level REST Metrics Harvester
We declare our automated test frameworks using versioned REST interface components to capture exact shuffle write bytes directly from the local Spark UI endpoint (`/api/v1/applications/<app-id>/stages`).


In [ ]:
from pyspark.sql import SparkSession
import time
import json
from urllib.request import urlopen
from urllib.parse import quote, urlparse
import os

def _spark_ui_candidates(sc):
    ui_url = getattr(sc, "uiWebUrl", None)
    if callable(ui_url): ui_url = ui_url()
    if not ui_url:
        try:
            scala_opt = sc._jsc.sc().uiWebUrl()
            if scala_opt.isDefined(): ui_url = scala_opt.get()
        except:
            ui_url = None
    if not ui_url: return []
    ui_url = ui_url.rstrip("/")
    candidates = [ui_url]
    parsed = urlparse(ui_url)
    if parsed.port and parsed.hostname not in {"localhost", "127.0.0.1"}:
        candidates.append(f"{parsed.scheme or 'http'}://127.0.0.1:{parsed.port}")
    return list(dict.fromkeys(candidates))

def extract_total_shuffle_metrics(spark, require_completed_stage=False):
    sc = spark.sparkContext
    app_id = quote(sc.applicationId, safe="/")
    candidates = _spark_ui_candidates(sc)
    
    for _ in range(20):
        for base_url in candidates:
            endpoint = f"{base_url}/api/v1/applications/{app_id}/stages?status=complete"
            try:
                with urlopen(endpoint, timeout=5) as resp:
                    stages = json.loads(resp.read().decode("utf-8"))
                latest_by_stage = {}
                for stage in stages:
                    sid = int(stage.get("stageId", -1))
                    aid = int(stage.get("attemptId", 0))
                    current = latest_by_stage.get(sid)
                    if current is None or aid > int(current.get("attemptId", 0)):
                        latest_by_stage[sid] = stage
                
                total_bytes = sum(int(st.get("shuffleWriteBytes", 0) or 0) for st in latest_by_stage.values())
                if require_completed_stage and len(latest_by_stage) == 0: break
                return total_bytes
            except:
                pass
        time.sleep(0.25)
    return 0

# ----------------------------------------------------------------------
# MadLava JVM bootstrap
# ----------------------------------------------------------------------
# The Java agent must be present on the command that launches PySpark's
# gateway JVM. The shared JSON is therefore attached through
# PYSPARK_SUBMIT_ARGS before any SparkContext/SparkSession is created.

MADLAVA_JAR = os.path.abspath("madlava-agent-0.1.0.jar").replace("\\", "/")
MADLAVA_CONFIG = os.path.abspath("madlava.json").replace("\\", "/")
MADLAVA_REPORTS = {}

for _path in (MADLAVA_JAR, MADLAVA_CONFIG):
    if not os.path.isfile(_path):
        raise FileNotFoundError(f"Missing required MadLava file: {_path}")

_MADLAVA_AGENT_OPTION = (
    f"-javaagent:{MADLAVA_JAR}=config={MADLAVA_CONFIG}"
)

_existing_submit_args = os.environ.get("PYSPARK_SUBMIT_ARGS", "").strip()

# Remove the shell marker so our --conf is inserted before it.
if _existing_submit_args.endswith("pyspark-shell"):
    _existing_submit_args = _existing_submit_args[:-len("pyspark-shell")].strip()

if "--driver-java-options" in _existing_submit_args:
    raise RuntimeError(
        "PYSPARK_SUBMIT_ARGS already defines --driver-java-options. "
        "Restart the kernel after removing that conflicting definition."
    )

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    f'{_existing_submit_args} '
    f'--driver-java-options "{_MADLAVA_AGENT_OPTION}" '
    f'pyspark-shell'
).strip()

from pyspark import SparkContext

def _gateway_madlava_available():
    if SparkContext._gateway is None:
        return False
    try:
        api = SparkContext._gateway.jvm.com.madlava.api.MadLavaStatistics
        return bool(api.isAvailable())
    except Exception:
        return False


_AGENT_BOOTSTRAPPED = _gateway_madlava_available()
_MADLAVA_DRIVER_PID = None

if SparkContext._gateway is not None and not _AGENT_BOOTSTRAPPED:
    raise RuntimeError(
        "A PySpark gateway JVM already exists without MadLava attached. "
        "Restart the kernel, keep the agent JAR and JSON beside the notebook, "
        "then Run All."
    )


class MadLavaScopeReports:
    """Thin Py4J adapter over MadLava's public scope/report APIs."""

    def __init__(self, spark):
        self.jvm = spark.sparkContext._jvm
        self.statistics = self.jvm.com.madlava.api.MadLavaStatistics
        self.scopes = self.jvm.com.madlava.api.MadLavaScopes
        self.reports = self.jvm.com.madlava.api.MadLavaReport

        if not bool(self.statistics.isAvailable()):
            raise RuntimeError(
                "MadLava is not available in the PySpark gateway JVM. "
                "Restart the kernel and verify the MadLava gateway launch configuration."
            )
        if not bool(self.scopes.isAvailable()):
            raise RuntimeError("MadLavaScopes.isAvailable() returned false.")

    def begin_scope(self, name):
        scope_id = str(self.scopes.beginScope(name))
        if not scope_id:
            raise RuntimeError(f"MadLava returned an empty scope ID for {name!r}.")
        return scope_id

    def end_scope(self, scope_id):
        result_id = str(self.scopes.endScope(scope_id))
        if not result_id:
            raise RuntimeError(
                f"MadLava returned an empty ScopeResult ID for {scope_id!r}."
            )
        return result_id

    def report_text(self, result_id):
        report = str(self.reports.scopeReportText(result_id))
        if not report.strip():
            raise RuntimeError(
                f"MadLava returned an empty report for {result_id!r}."
            )
        return report


def _madlava_driver_pid(spark):
    return int(
        spark.sparkContext._jvm.java.lang.ProcessHandle.current().pid()
    )


def run_with_madlava_scope(scope_name, spark, workload, *args, **kwargs):
    """
    Run the existing lab workload unchanged inside one MadLava scope.

    Spark's own metrics remain the lab's primary measurements. MadLava only
    adds the JVM-level evidence printed immediately after the workload.
    """
    madlava = MadLavaScopeReports(spark)
    scope_id = madlava.begin_scope(scope_name)
    print(f"🌋 MadLava scope started: {scope_name} ({scope_id})")

    result = None
    workload_error = None
    workload_traceback = None

    try:
        result = workload(*args, **kwargs)
    except BaseException as exc:
        workload_error = exc
        workload_traceback = exc.__traceback__

    try:
        result_id = madlava.end_scope(scope_id)
        report = madlava.report_text(result_id)
        MADLAVA_REPORTS[scope_name] = report
        print(f"\n🌋 MadLava under-the-hood report: {scope_name}")
        print(report)
    except Exception as report_error:
        if workload_error is None:
            raise
        print(
            f"⚠️ MadLava report collection also failed after the workload error: "
            f"{report_error}"
        )

    if workload_error is not None:
        raise workload_error.with_traceback(workload_traceback)

    return result

print("✅ REST metric diagnostics framework successfully calibrated!")


### Step 2: Define Lifecycle Actions and the Object Graph Generators
We declare our runner routines. `reset_and_build_spark` manages isolated session configurations. `execute_graph_probe` can generate either **Workload A** (inserting the exact same 5KB object pointer 10 times inside a single list row) or **Workload B** (distributing a single 5KB object reference across 5,000 separate independent row allocations).


In [ ]:
import random

MORTUARY_RANDOM_SEED = 42

def reset_and_build_spark(mode="java", tracking="true", register_classes=False):
    global _AGENT_BOOTSTRAPPED, _MADLAVA_DRIVER_PID
    active_session = SparkSession.getActiveSession()
    if active_session is not None:
        print("⚰️ Parking old Spark instance...")
        active_session.stop()
        time.sleep(2)
        
    builder = SparkSession.builder.master("local[*]").appName(f"kryo-internals-{mode}-{tracking}")
    builder.config("spark.driver.memory", "4g")
    

    if mode == "kryo":
        builder.config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        builder.config("spark.kryo.referenceTracking", tracking)
        if register_classes:
            print(f"🚀 Booting Kryo context | Tracking: {tracking} | Class Registration: ACTIVE...")
            builder.config("spark.kryo.classesToRegister", "java.util.ArrayList")
        else:
            print(f"🚀 Booting Kryo context | Tracking: {tracking} | Class Registration: NONE...")
    else:
        print("📦 Booting context with standard Java serialization defaults...")
        builder.config("spark.serializer", "org.apache.spark.serializer.JavaSerializer")
        
    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel("WARN")

    if not _gateway_madlava_available():
        runtime_args = [
            str(arg)
            for arg in spark.sparkContext._jvm.java.lang.management
                .ManagementFactory.getRuntimeMXBean()
                .getInputArguments()
        ]
        javaagent_args = [
            arg for arg in runtime_args if arg.startswith("-javaagent:")
        ]
        raise RuntimeError(
            "MadLava did not activate in the PySpark gateway JVM. "
            f"Observed JVM -javaagent arguments: {javaagent_args or ['<none>']}. "
            "If the MadLava argument is present, inspect JVM startup stderr for "
            "'bootstrap disabled'; that indicates the agent rejected its startup "
            "configuration. Restart the kernel after correcting the cause."
        )

    driver_pid = _madlava_driver_pid(spark)
    if _MADLAVA_DRIVER_PID is None:
        _MADLAVA_DRIVER_PID = driver_pid
    elif driver_pid != _MADLAVA_DRIVER_PID:
        raise RuntimeError(
            f"Expected one persistent MadLava JVM, but PID changed "
            f"from {_MADLAVA_DRIVER_PID} to {driver_pid}."
        )
    _AGENT_BOOTSTRAPPED = True
    return spark

def execute_graph_probe(spark, target_workload="intra_record"):
    sc = spark.sparkContext
    print(f"  ⚡ Building Graph Topology: {target_workload.upper()} via Py4J Gateway...")
    
    # Anchor data generation to the exact global seed sequence across consecutive runs
    rng = random.Random(MORTUARY_RANDOM_SEED)
    
    # Construct a seed-locked heavy string carrying true variable data entropy noise
    base_padding = "mortuary_heavy_payload_string_shroud_padding_tax_"
    noise_sequence = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz0123456789") for _ in range(500))
    heavy_padding = (base_padding * 90) + noise_sequence
    
    # Instantiate a single 5KB heavy metadata block inside the JVM heap
    shared_node = sc._jvm.java.util.ArrayList()
    shared_node.add(heavy_padding)
    
    master_list = sc._jvm.java.util.ArrayList()
    
    if target_workload == "intra_record":
        # Workload A: Add the exact same object reference 10 times INSIDE each record container
        for i in range(5000):
            record_graph = sc._jvm.java.util.ArrayList()
            record_graph.add(int(i))
            for _ in range(10):
                record_graph.add(shared_node)
            master_list.add(record_graph)
    else:
        # Workload B: Add the shared reference exactly once per record across 5,000 independent row records
        for i in range(5000):
            record_graph = sc._jvm.java.util.ArrayList()
            record_graph.add(int(i))
            record_graph.add(shared_node)
            master_list.add(record_graph)
            
    # Parallelize into an unoptimized heap JavaRDD
    java_rdd = sc._jsc.parallelize(master_list, 4)
    
    start_bytes = extract_total_shuffle_metrics(spark)
    start_time = time.perf_counter()
    
    print("  🚀 Shuffling object layout across partitions...")
    shuffled_rdd = java_rdd.repartition(16)
    record_count = shuffled_rdd.count()
    
    duration = time.perf_counter() - start_time
    end_bytes = extract_total_shuffle_metrics(spark, require_completed_stage=True)
    shuffle_bytes = end_bytes - start_bytes
    
    print(f"  ├─ Isolated Shuffle Size : {shuffle_bytes:,} bytes")
    print(f"  └─ Processing Execution  : {duration:.2f} seconds")
    return shuffle_bytes


### Symmetrical Benchmarking Preparation: The JVM Warmup Phase
To isolate true framework serialization footprints fairly, we spin up a mock context using the exact same Py4J RDD path as the tests. This forces the background JVM daemon to load the `java.util.ArrayList` class maps, stabilize ClassLoaders, and warm up the RDD repartition JIT compiler using the exact same seed parameters so Phase 1 doesn't shoulder the initial boot overhead.


In [ ]:
print("🔥 TRIGGERING COLD JVM WARMUP RUN...")
warmup_session = reset_and_build_spark("warmup")
sc_warm = warmup_session.sparkContext

print("⚡ Fetching Py4J targets and warming JVM RDD class pools...")
rng_warm = random.Random(MORTUARY_RANDOM_SEED)
warm_list = sc_warm._jvm.java.util.ArrayList()

# Generate a seed-locked mock RDD payload mirroring the test data architecture
base_padding = "warmup_heavy_payload_string_shroud_padding_tax_"
noise_sequence = "".join(rng_warm.choice("abcdefghijklmnopqrstuvwxyz0123456789") for _ in range(500))
heavy_padding = (base_padding * 90) + noise_sequence

shared_warm_node = sc_warm._jvm.java.util.ArrayList()
shared_warm_node.add(heavy_padding)

for i in range(500):
    mock_record = sc_warm._jvm.java.util.ArrayList()
    mock_record.add(int(i))
    mock_record.add(shared_warm_node)
    warm_list.add(mock_record)

# Force a parallelized heap shuffle to execute JIT compilations across the serializer classes
warm_rdd = sc_warm._jsc.parallelize(warm_list, 4)
warm_rdd.repartition(16).count()

print("✅ JVM Heap and RDD ClassLoader layers successfully stabilized.")


# Part I: Isolating Intra-Record Deduplication (Workload A)


In [ ]:
print("=== PHASE 1: INTRA-RECORD RUN WITH JAVA BASELINE ===")
spark_java = reset_and_build_spark(mode="java")
java_intra = run_with_madlava_scope("lab2_java_intra_record", spark_java, execute_graph_probe, spark_java, "intra_record")


In [ ]:
print("\n=== PHASE 2: INTRA-RECORD RUN WITH BLIND KRYO (NO TRACKING) ===")
spark_blind = reset_and_build_spark(mode="kryo", tracking="false")
blind_intra = run_with_madlava_scope("lab2_kryo_tracking_off_intra_record", spark_blind, execute_graph_probe, spark_blind, "intra_record")


In [ ]:
print("\n=== PHASE 3: INTRA-RECORD RUN WITH TRACKED KRYO (TRACKING ACTIVE) ===")
spark_tracked = reset_and_build_spark(mode="kryo", tracking="true")
tracked_intra = run_with_madlava_scope("lab2_kryo_tracking_on_intra_record", spark_tracked, execute_graph_probe, spark_tracked, "intra_record")


# Part II: Exposing the Inter-Record No-Deduplication Illusion (Workload B)


In [ ]:
print("=== PHASE 4: INTER-RECORD RUN WITH BLIND KRYO (NO TRACKING) ===")
spark_blind_inter = reset_and_build_spark(mode="kryo", tracking="false")
blind_inter = run_with_madlava_scope("lab2_kryo_tracking_off_inter_record", spark_blind_inter, execute_graph_probe, spark_blind_inter, "inter_record")


In [ ]:
print("\n=== PHASE 5: INTER-RECORD RUN WITH TRACKED KRYO (TRACKING ACTIVE) ===")
spark_tracked_inter = reset_and_build_spark(mode="kryo", tracking="true")
tracked_inter = run_with_madlava_scope("lab2_kryo_tracking_on_inter_record", spark_tracked_inter, execute_graph_probe, spark_tracked_inter, "inter_record")


### Step 5: Final Telemetry Diagnostics Summary
We review the extracted payload numbers from both workloads to map out the native boundaries of the state tracking engine.


In [ ]:
print("\n📊 --- MORTUARY LAB 3 DIAGNOSTIC METRICS SUMMARY ---")
print("📦 WORKLOAD A: INTRA-RECORD DEPENDENCIES (10 Pointers inside 1 row)")
print(f"  ├─ Java Baseline Bytes     : {java_intra:,} bytes")
print(f"  ├─ Blind Kryo Bytes        : {blind_intra:,} bytes")
print(f"  └─ Tracked Kryo Bytes      : {tracked_intra:,} bytes")

print("\n📦 WORKLOAD B: INTER-RECORD DEPENDENCIES (1 Pointer shared across 5,000 rows)")
print(f"  ├─ Blind Kryo Inter Bytes  : {blind_inter:,} bytes")
print(f"  └─ Tracked Kryo Inter Bytes: {tracked_inter:,} bytes")

intra_saved = blind_intra - tracked_intra
inter_variance = abs(blind_inter - tracked_inter)

print(f"\n⚰️ Forensic Verdict:")
print(f"  ├─ Inside a single row, Reference Tracking saved {intra_saved:,} duplicate bytes!")
print(f"  └─ Across multiple rows, Reference Tracking variance was exactly {inter_variance} bytes!")
print("\n🔥 Mathematical Proof: Kryo completely resets its state lookup table between records.")
print("Tracking provides zero deduplication benefits across separate rows. It is a local constraint, not a global database cache.")

SparkSession.getActiveSession().stop()
print("\n💀 SparkContext destroyed. Grid safely parked.")


## 📊 Post-Lab Analysis: Mapping the Tracking Boundaries

This diagnostics lab unmasks the exact architectural scope and lifecycle constraints governing Kryo's state-tracking mechanisms.

### 1. The Workload A Triumph: Intra-Record Compression
Inside a single record graph, reference tracking delivers massive optimization. When tracking is explicitly disabled (`spark.kryo.referenceTracking=false`), Kryo acts without a short-term layout memory. It deep-copies the heavy 5KB text payload 10 separate times per row, bloating the network shuffle footprint to **1,878,942 bytes**.

Activating tracking allows Kryo to maintain an internal object dictionary during the serialization window of that specific record graph. It encodes the 5KB block exactly once; for the remaining 9 entries, it intercepts the pointer and substitutes it with light 4-byte reference tokens, slashing the final payload to **254,380 bytes**—saving **1.62 megabytes** of empty transit weight.

### 2. The Workload B Illusion: Inter-Record Reset Boundaries
When the exact same 5KB metadata object pointer is shared *across* 5,000 separate, independent rows, the deduplication magic completely flatlines. Blind Kryo ships out **241,298 bytes**, while Tracked Kryo writes out **243,906 bytes**. Tracked Kryo fails to compress the cross-row reference, actually losing by exactly **2,608 bytes**.

This outcome exposes the absolute operational boundary of the framework: because Apache Spark streams and serializes shuffle records as entirely independent items, **Kryo completely flushes and destroys its reference tracking tables at the boundary of every single row record.** The extra 2,608 bytes recorded by the telemetry is the literal 'bookkeeping tax'—the processing overhead the JVM wasted allocating and executing tracking lookups for thousands of records that can never share an inheritance.

### 3. The Java Baseline Paradox
Advanced systems engineers will notice that standard Java serialization handled Workload A remarkably well, writing only **76,664 bytes** without any custom configuration. Unlike Kryo, Java's `ObjectOutputStream` has graph-wide reference tracking hardcoded into its baseline specification. It maintains an unbroken graph handle memory registry for its entire stream lifecycle unless manually forced to clear via a `.reset()` call. While Java serialization remains highly CPU-inefficient due to legacy runtime reflection requirements, its fundamental layout specification natively mitigates intra-record object duplication hazards out of the box.
